# 02 - Data Cleaning

Runs the cleaning pipeline (`src/data_cleaning.py`) end to end and shows
the before/after effect of each documented decision. The full rationale
for each decision is in `reports/data_quality_report.md`; this notebook
demonstrates it against the live data rather than repeating the prose.

In [1]:
import sys
sys.path.insert(0, "..")
import json
import pandas as pd
from src.data_cleaning import run_pipeline

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

reports = run_pipeline()
print(json.dumps(reports, indent=2))

INFO src.data_cleaning: clean_coverage: {'rows_before': 37749, 'duplicate_rows_removed': 0, 'invalid_year_rows_removed': 0, 'invalid_coverage_rows_removed': 0, 'unknown_country_code_rows_removed': 0, 'rows_after': 37749}


INFO src.data_cleaning: clean_reported_cases: {'rows_before': 64860, 'duplicate_rows_removed': 0, 'invalid_year_rows_removed': 0, 'negative_case_rows_removed': 0, 'missing_cases_retained_as_null': 10135, 'unknown_country_code_rows_removed': 0, 'rows_after': 64860}


INFO src.data_cleaning: clean_population: {'rows_before': 9710, 'duplicate_rows_removed': 0, 'non_positive_population_rows_removed': 0, 'unknown_country_code_rows_removed': 0, 'rows_after': 9710}


INFO src.data_cleaning: derive_incidence_rate: {'rows_before': 64860, 'rows_dropped_no_population_match': 8612, 'rows_dropped_no_case_count': 7596, 'rows_after': 48652}


INFO src.data_cleaning: clean_vaccine_introduction: {'rows_before': 1565, 'duplicate_rows_removed': 0, 'invalid_who_region_rows_removed': 0, 'inconsistent_intro_yes_no_year_removed': 0, 'rows_after': 1565}


INFO src.data_cleaning: clean_vaccine_schedule: {'rows_before': 1824, 'duplicate_rows_removed': 0, 'invalid_who_region_rows_removed': 0, 'non_positive_rounds_removed': 0, 'rows_after': 1824}


{
  "coverage": {
    "rows_before": 37749,
    "duplicate_rows_removed": 0,
    "invalid_year_rows_removed": 0,
    "invalid_coverage_rows_removed": 0,
    "unknown_country_code_rows_removed": 0,
    "rows_after": 37749
  },
  "reported_cases": {
    "rows_before": 64860,
    "duplicate_rows_removed": 0,
    "invalid_year_rows_removed": 0,
    "negative_case_rows_removed": 0,
    "missing_cases_retained_as_null": 10135,
    "unknown_country_code_rows_removed": 0,
    "rows_after": 64860
  },
  "population": {
    "rows_before": 9710,
    "duplicate_rows_removed": 0,
    "non_positive_population_rows_removed": 0,
    "unknown_country_code_rows_removed": 0,
    "rows_after": 9710
  },
  "incidence_rate": {
    "rows_before": 64860,
    "rows_dropped_no_population_match": 8612,
    "rows_dropped_no_case_count": 7596,
    "rows_after": 48652
  },
  "vaccine_introduction": {
    "rows_before": 1565,
    "duplicate_rows_removed": 0,
    "invalid_who_region_rows_removed": 0,
    "inconsisten

## Reading the cleaning report

Each dataset's report shows `rows_before` / `rows_after` plus the count
removed by every individual rule. The most consequential decision is on
`reported_cases`: 10,135 rows have a missing `Cases` value and are kept as
null rather than dropped or filled with 0 (see rationale in the data
quality report).

In [2]:
from src import data_loader

cases = data_loader.load_reported_cases()
missing = cases["Cases"].isna()
print(f"Missing Cases: {missing.sum()} / {len(cases)} ({missing.mean():.1%})")
print()
print("Missing-rate by disease:")
print(cases.groupby("Disease")["Cases"].apply(lambda s: s.isna().mean()).sort_values(ascending=False))

Missing Cases: 10135 / 64860 (15.6%)

Missing-rate by disease:
Disease
TETANUS_NEONATAL    0.237285
TETANUS_TOTAL       0.185780
PERTUSSIS           0.175984
DIPHTHERIA          0.171801
RUBELLA             0.169853
MEASLES             0.105678
POLIO               0.000000
Name: Cases, dtype: float64


## Incidence rate derivation check

`Incidence_rate = Cases / Population * 100,000`. Spot-check a handful of
rows against the raw inputs.

In [3]:
incidence = pd.read_csv("../data/processed/clean_incidence_rate.csv")
population = data_loader.load_population()

sample = incidence.sample(5, random_state=1)
for _, row in sample.iterrows():
    pop_row = population[(population["Code"] == row["Code"]) & (population["Year"] == row["Year"])]
    pop_value = pop_row["Population"].iloc[0]
    expected = row["Cases"] if "Cases" in row else None
    recomputed = row["Denominator"]
    print(row["Code"], row["Year"], row["Disease"], "-> rate:", round(row["Incidence_rate"], 3),
          "| denominator matches population lookup:", abs(pop_value - row["Denominator"]) < 1)

GNB 1986 TETANUS_TOTAL -> rate: 27.847 | denominator matches population lookup: True
FSM 1989 POLIO -> rate: 0.0 | denominator matches population lookup: True
DNK 2023 MEASLES -> rate: 0.151 | denominator matches population lookup: True
CPV 1989 DIPHTHERIA -> rate: 0.0 | denominator matches population lookup: True
COD 2011 PERTUSSIS -> rate: 3.461 | denominator matches population lookup: True


## Cleaned outputs

Written to `data/processed/`:
- `clean_coverage.csv`
- `clean_reported_cases.csv`
- `clean_incidence_rate.csv` (derived)
- `clean_vaccine_introduction.csv`
- `clean_vaccine_schedule.csv`

In [4]:
from src.config import PROCESSED_FILES

for name, path in PROCESSED_FILES.items():
    df = pd.read_csv(path)
    print(f"{name:22s} {df.shape}")

coverage               (37749, 9)


reported_cases         (64860, 7)


incidence_rate         (48652, 8)
vaccine_introduction   (1565, 6)
vaccine_schedule       (1824, 12)
